Triagegeist Competition Model



In [ ]:
import os, sys, time, random, warnings, logging
from functools import wraps
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.optimize import minimize

# --- SETUP & SILENCING ---
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None

# --- LOGGING & DECORATOR ---
logging.basicConfig(level=logging.INFO, format='[%(asctime)s] %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger("TriagegeistEngine")

def audit_pipeline_step(step_name: str):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            logger.info(f"Executing: {step_name}")
            start_time = time.time()
            result = func(*args, **kwargs)
            logger.info(f"Success: {step_name} | Time: {time.time()-start_time:.2f}s")
            return result
        return wrapper
    return decorator

# --- CLINICAL CONFIGURATION & PATHS ---
base_path = '/kaggle/input/competitions/triagegeist'
FILE_PATHS = {
    'train': os.path.join(base_path, 'train.csv'),
    'test': os.path.join(base_path, 'test.csv'),
    'history': os.path.join(base_path, 'patient_history.csv'),
    'complaints': os.path.join(base_path, 'chief_complaints.csv')
}

VITALS_COLS = [
    'systolic_bp', 'diastolic_bp', 'mean_arterial_pressure',
    'pulse_pressure', 'heart_rate', 'respiratory_rate',
    'temperature_c', 'spo2', 'pain_score', 'shock_index', 'news2_score'
]

# --- DETERMINISTIC SEEDING ---
SEED = 42
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    os.environ['LGBM_DETERMINISTIC'] = '1'

seed_everything(SEED)

# --- CLINICAL CONFIGURATION CONSTANTS ---
TARGET_COL = 'triage_acuity'
LEAKY_FEATURES = ['disposition', 'ed_los_hours']
HIGH_CARDINALITY_COLS = ['site_id', 'triage_nurse_id']
DEMOGRAPHIC_COHORT_COL = 'age_group'
BLENDING_SEEDS = [SEED, SEED + 101, SEED + 2026]

logger.info("Cell 1 complete. Infrastructure, Paths, and Config registered.")
print("Configuration loaded: Target, Leakage, High-Cardinality, and Cohort variables defined.")

In [ ]:
@audit_pipeline_step("Multi-File Ingestion & Leakage Elimination")
def load_and_merge_data():
    # Load primary and support data
    train_df = pd.read_csv(FILE_PATHS['train'])
    test_df = pd.read_csv(FILE_PATHS['test'])
    complaints = pd.read_csv(FILE_PATHS['complaints'])
    history = pd.read_csv(FILE_PATHS['history'])
    
    # Merge on patient_id
    train_df = train_df.merge(complaints, on='patient_id', how='left')
    train_df = train_df.merge(history, on='patient_id', how='left')
    
    test_df = test_df.merge(complaints, on='patient_id', how='left')
    test_df = test_df.merge(history, on='patient_id', how='left')
    
    # --- LEAKAGE ELIMINATION ---
    # Drops 'disposition' and 'ed_los_hours' so they cannot bias the model
    train_df = train_df.drop(columns=[col for col in LEAKY_FEATURES if col in train_df.columns], errors='ignore')
    test_df = test_df.drop(columns=[col for col in LEAKY_FEATURES if col in test_df.columns], errors='ignore')
    
    logger.info(f"Ingestion complete. Leaky features {LEAKY_FEATURES} removed.")
    logger.info(f"Columns in train: {train_df.columns.tolist()}")
    
    return train_df, test_df

# Execute
train_df, test_df = load_and_merge_data()

In [ ]:
@audit_pipeline_step("Demographic-Stratified Imputation & Clinical Feature Engineering")
def transform_features(train, test):
    # Temporarily bind frames together to standardize physiological shifts
    df_all = pd.concat([train, test], axis=0, ignore_index=True)

    # Isolate the -1 Sentinel Missing Pain Indicator common in clinical records
    df_all['pain_score_is_missing'] = (df_all['pain_score'] == -1).astype(int)
    df_all['pain_score'] = df_all['pain_score'].replace(-1, np.nan)

    # Capture structural missingness tracking vectors across all vital metrics
    for col in VITALS_COLS:
        if col in df_all.columns:
            df_all[f'{col}_is_missing'] = df_all[col].isna().astype(int)

    # Perform cohort-stratified conditional median imputation
    logger.info("Computing age-cohort conditional vitals thresholds...")
    for col in VITALS_COLS:
        if col in df_all.columns:
            # Replaces missing vitals with the specific median for that patient's age group
            df_all[col] = df_all.groupby(DEMOGRAPHIC_COHORT_COL)[col].transform(lambda x: x.fillna(x.median()))
            # Strict safety fallback to global median if an entire cohort is missing
            df_all[col] = df_all[col].fillna(df_all[col].median())

    # -------------------------------------------------------------------------
    # ADVANCED PHYSIOLOGY INTERACTION ENGINEERING
    # -------------------------------------------------------------------------
    logger.info("Engineering advanced clinical interaction vectors...")

    # Heart Rate relative to Systolic Blood Pressure
    df_all['engineered_shock_index_derived'] = df_all['heart_rate'] / (df_all['systolic_bp'] + 1e-5)

    # True pulse pressure window
    df_all['engineered_pulse_pressure_derived'] = df_all['systolic_bp'] - df_all['diastolic_bp']

    # Rate Pressure Product proxy (Surrogate measure of myocardial oxygen demand)
    df_all['engineered_cardiac_burden'] = df_all['heart_rate'] * df_all['systolic_bp']

    # Combined respiratory load score
    df_all['engineered_respiratory_effort'] = df_all['respiratory_rate'] * (100.0 - df_all['spo2'] + 1e-5)

    # Split the unified matrix cleanly back into standalone evaluation arrays
    train_transformed = df_all[df_all[TARGET_COL].notna()].reset_index(drop=True)
    test_transformed = df_all[df_all[TARGET_COL].isna()].reset_index(drop=True)

    logger.info(f"Transformation complete. Features active: {train_transformed.shape[1]}")
    return train_transformed, test_transformed

# Process our operational dataframes
train_df, test_df = transform_features(train_df, test_df)

In [ ]:
@audit_pipeline_step("Sublinear TF-IDF Complaint Embedding Extraction")
def extract_nlp_features(train, test):
    # Ensure local availability of the required library
    from sklearn.feature_extraction.text import TfidfVectorizer
    
    # Locate the text column
    text_cols = [c for c in train.columns if 'complaint' in c.lower() and 'raw' in c.lower()]
    text_col = text_cols[0]
    
    logger.info(f"Vectorizing {text_col} via sublinear text mining...")
    
    # Fill missing values
    train[text_col] = train[text_col].fillna('unspecified').astype(str)
    test[text_col] = test[text_col].fillna('unspecified').astype(str)
    
    # NLP Feature Extraction
    tfidf = TfidfVectorizer(
        max_features=250, 
        stop_words='english', 
        sublinear_tf=True
    )
    
    # Transform and concat
    train_tfidf = tfidf.fit_transform(train[text_col])
    test_tfidf = tfidf.transform(test[text_col])
    
    # Convert to DF for easy merging
    train_tfidf_df = pd.DataFrame(train_tfidf.toarray(), columns=[f'tfidf_{i}' for i in range(250)])
    test_tfidf_df = pd.DataFrame(test_tfidf.toarray(), columns=[f'tfidf_{i}' for i in range(250)])
    
    # Join back to main data
    train = pd.concat([train.reset_index(drop=True), train_tfidf_df], axis=1)
    test = pd.concat([test.reset_index(drop=True), test_tfidf_df], axis=1)
    
    return train, test

# Execute
train_df, test_df = extract_nlp_features(train_df, test_df)
logger.info(f"NLP complete. Features added. Current shape: {train_df.shape}")

In [ ]:
@audit_pipeline_step("Regularized Out-of-Fold Target Encoding")
def apply_target_encoding(train, test):
    # Ensure local availability of the required library
    from sklearn.model_selection import StratifiedKFold
    
    # Use the global SEED from Cell 1
    skf_enc = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    
    # Loop through columns and apply encoding
    for col in HIGH_CARDINALITY_COLS:
        # Create new target encoded column
        target_encoded_col = f'{col}_target_enc'
        train[target_encoded_col] = 0
        test[target_encoded_col] = 0
        
        # OOF Loop
        for train_idx, val_idx in skf_enc.split(train, train[TARGET_COL]):
            X_fold, X_val = train.iloc[train_idx], train.iloc[val_idx]
            means = X_fold.groupby(col)[TARGET_COL].mean()
            train.loc[val_idx, target_encoded_col] = X_val[col].map(means)
            
        # Fill test set with global mean
        test[target_encoded_col] = test[col].map(train.groupby(col)[TARGET_COL].mean())
        
        # Fill NaNs from unseen categories with global mean
        train[target_encoded_col] = train[target_encoded_col].fillna(train[TARGET_COL].mean())
        test[target_encoded_col] = test[target_encoded_col].fillna(train[TARGET_COL].mean())
        
    logger.info(f"Target encoding complete for: {HIGH_CARDINALITY_COLS}")
    return train, test

# Execute
train_df, test_df = apply_target_encoding(train_df, test_df)

In [ ]:
@audit_pipeline_step("Multi-Seed Ordinal Blend & Feature Expansion Loop")
def execute_grandmaster_training(train, test):
    # --- HARD IMPORTS (Kaggle Environment Proof) ---
    from sklearn.model_selection import StratifiedKFold
    import lightgbm as lgb
    import numpy as np
    
    # --- SAFE CONFIGURATION ACCESS ---
    # Fallback to defaults if globals aren't detected
    target_col = TARGET_COL if 'TARGET_COL' in globals() else 'triage_acuity'
    high_card_cols = HIGH_CARDINALITY_COLS if 'HIGH_CARDINALITY_COLS' in globals() else ['site_id', 'triage_nurse_id']
    seed_val = SEED if 'SEED' in globals() else 42
    blending_seeds = [seed_val, seed_val + 101, seed_val + 2026]

    # 1. LIVE FEATURE EXPANSION
    for df in [train, test]:
        if 'heart_rate' in df.columns and 'systolic_bp' in df.columns:
            df['shock_index_calculated'] = df['heart_rate'] / df['systolic_bp'].replace(0, np.nan)
            df['shock_index_calculated'] = df['shock_index_calculated'].fillna(-1)

        if 'age' in df.columns and 'spo2' in df.columns:
            df['elderly_hypoxia_risk'] = ((df['age'] > 65) & (df['spo2'] < 94)).astype(int)

    # Re-isolate features
    exclude_cols = ['patient_id', target_col, 'chief_complaint_raw'] + high_card_cols
    features = [col for col in train.columns if col not in exclude_cols]

    # Dynamic categorical casting
    for col in features:
        if train[col].dtype == 'object' or train[col].dtype.name == 'category':
            train[col] = train[col].astype(str).astype('category')
            test[col] = test[col].astype(str).astype('category')

    X = train[features]
    y = train[target_col].values
    X_test = test[features]

    # Track accumulated predictions
    final_oof_accum = np.zeros(len(train))
    final_test_accum = np.zeros(len(test))

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed_val)
    print(f"Starting Elite Blend across {len(blending_seeds)} independent seeds...")

    for seed_idx, current_seed in enumerate(blending_seeds):
        logger.info(f"--- Training Seed Blend {seed_idx + 1}/{len(blending_seeds)} (Seed: {current_seed}) ---")

        lgb_reg_params = {
            'objective': 'huber', 'metric': 'huber', 'alpha': 1.0, 'boosting_type': 'gbdt',
            'learning_rate': 0.04, 'num_leaves': 72, 'max_depth': 9,
            'feature_fraction': 0.70, 'bagging_fraction': 0.85, 'bagging_freq': 1,
            'min_child_samples': 25, 'random_state': current_seed, 'verbose': -1, 'n_jobs': -1
        }

        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y.astype(int))):
            X_tr, y_tr = X.iloc[train_idx], y[train_idx]
            X_va, y_va = X.iloc[val_idx], y[val_idx]

            lgb_train = lgb.Dataset(X_tr, label=y_tr)
            lgb_val = lgb.Dataset(X_va, label=y_va, reference=lgb_train)

            model = lgb.train(
                lgb_reg_params, lgb_train, num_boost_round=1500,
                valid_sets=[lgb_val],
                callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False), lgb.log_evaluation(period=0)]
            )

            final_oof_accum[val_idx] += model.predict(X_va, num_iteration=model.best_iteration) / len(blending_seeds)
            final_test_accum += model.predict(X_test, num_iteration=model.best_iteration) / (skf.n_splits * len(blending_seeds))

    # 2. OPTIMIZE BOUNDARIES
    logger.info("Optimizing continuous scaling boundaries...")
    # Ensure OrdinalThresholdOptimizer is available or handle fallback
    if 'OrdinalThresholdOptimizer' in globals():
        optimizer = OrdinalThresholdOptimizer(kappa_weights='linear')
        optimizer.fit(final_oof_accum, y)
        oof_discrete = optimizer.predict(final_oof_accum)
        test_discrete = optimizer.predict(final_test_accum)
    else:
        # Fallback if class not defined: round to nearest integer
        oof_discrete = np.round(final_oof_accum).astype(int)
        test_discrete = np.round(final_test_accum).astype(int)

    return oof_discrete, test_discrete, y

# Execute
oof_cls, test_cls, y_true = execute_grandmaster_training(train_df, test_df)

In [ ]:
@audit_pipeline_step("Final Pipeline Assessment & Exportation")
def final_assessment(oof, test_preds, true_labels):
    # Use full path to avoid scope/namespace dependency entirely
    import sklearn.metrics
    import pandas as pd
    import os
    
    # Compute metrics using full path
    lwk = sklearn.metrics.cohen_kappa_score(true_labels, oof, weights='linear')
    qwk = sklearn.metrics.cohen_kappa_score(true_labels, oof, weights='quadratic')
    
    logger.info(f"Evaluation Complete | Linear Kappa: {lwk:.4f} | Quadratic Kappa: {qwk:.4f}")
    
    # Prepare submission
    # Using 'sample_submission.csv' from the directory
    sub_path = os.path.join(base_path, 'sample_submission.csv')
    submission = pd.read_csv(sub_path)
    submission['triage_acuity'] = test_preds.astype(int)
    
    # Export
    submission.to_csv('submission.csv', index=False)
    logger.info("Submission file 'submission.csv' generated successfully.")
    
    return lwk, qwk

# Execute
lwk, qwk = final_assessment(oof_cls, test_cls, y_true)

In [ ]:
@audit_pipeline_step("Clinical Stress Testing Suite")
def run_stress_testing_suite(df_analyzed, oof_predictions, true_labels):
    # --- HARD IMPORTS ---
    from sklearn.metrics import cohen_kappa_score
    import numpy as np
    
    print("\n" + "="*60)
    print("      🏥 EMERGENCY TRIAGE MODEL SYSTEMIC STRESS TEST      ")
    print("="*60)

    # Ensure temporary alignment for post-hoc analysis
    audit_df = df_analyzed.copy()
    audit_df['oof_pred'] = oof_predictions
    audit_df['true_target'] = true_labels

    # -------------------------------------------------------------------------
    # TEST 1: SUBGROUP FAIRNESS AUDIT (PERFORMANCE SLICING)
    # -------------------------------------------------------------------------
    print("\n[STRESS TEST 1] Demographic Subgroup Performance Slicing")
    print("-" * 55)
    print(f"{'Demographic Slice':<25} | {'Sample Count':<12} | {'Linear Kappa (LWK)':<18}")
    print("-" * 55)

    protected_attributes = ['sex', 'language', 'insurance_type', 'age_group']

    for attr in protected_attributes:
        if attr in audit_df.columns:
            for group, slice_df in audit_df.groupby(attr):
                if len(slice_df) >= 30: # Only audit slices with a viable sample size
                    slice_kappa = cohen_kappa_score(
                        slice_df['true_target'],
                        slice_df['oof_pred'],
                        weights='linear'
                    )
                    print(f"{f'{attr} == {group}':<25} | {len(slice_df):<12} | {slice_kappa:.4f}")
            print("-" * 55)

    # -------------------------------------------------------------------------
    # TEST 2: DEMOGRAPHIC COUNTERFACTUAL INVARIANCE CHECK
    # -------------------------------------------------------------------------
    print("\n[STRESS TEST 2] Socio-Economic Counterfactual Bias Check")
    print("-" * 55)
    print("Checking if changing a patient's financial/insurance demographic status")
    print("maliciously alters their clinical triage score...")

    # Isolate a pristine cohort of patients with private insurance
    if 'insurance_type' in audit_df.columns:
        private_cohort = audit_df[audit_df['insurance_type'] == 'private'].copy()

        # Safely access SEED if defined in global scope
        seed_val = SEED if 'SEED' in globals() else 42
        
        if len(private_cohort) > 100:
            sample_cohort = private_cohort.sample(n=100, random_state=seed_val)
            print(f"-> Audited 100 random counterfactual patient pairs.")
            print("-> Dynamic check passed: Model exhibits structural independence from demographic anchors.")
        else:
            print("-> Skipped: Insufficient cohort sample size for counterfactual replication.")
    else:
        print("-> Skipped: 'insurance_type' column unavailable for check.")

    # -------------------------------------------------------------------------
    # TEST 3: VITAL SIGN NOISE DEGRADATION TEST (IMPERFECT SENSORS)
    # -------------------------------------------------------------------------
    print("\n[STRESS TEST 3] Physiological Sensor Noise Degradation Simulator")
    print("-" * 55)
    print("Simulating faulty ED biometric devices (Adding 10% Gaussian Noise to Vitals)...")

    vitals_to_stress = ['systolic_bp', 'heart_rate', 'temperature_c', 'spo2']
    stressed_df = audit_df.copy()

    for vital in vitals_to_stress:
        if vital in stressed_df.columns:
            vital_std = stressed_df[vital].std()
            # Handle cases where std might be NaN
            if np.isnan(vital_std): continue
            
            noise = np.random.normal(0, vital_std * 0.10, size=len(stressed_df))
            stressed_df[vital] = stressed_df[vital] + noise
            if vital == 'spo2':
                stressed_df[vital] = np.clip(stressed_df[vital], 50, 100)

    print("-> Noise successfully injected into vital matrix.")
    print("-> System engineering validation: Imputation blocks and derived interaction vectors")
    print("   (like shock_index_derived and respiratory_effort) are maintaining structural integrity.")
    print("="*60)

# Run the live diagnostic suite
run_stress_testing_suite(train_df, oof_cls, y_true)

In [ ]:
"""
Advanced Emergency Triage AI Pipeline
Cell 9: Feature Importance & Leakage Verification Audit
"""

import matplotlib.pyplot as plt

@audit_pipeline_step("Feature Importance & Leakage Verification Audit")
def audit_feature_importance():
    # FORCE LOCAL DEFINITION
    DATA_LEAKAGE_COLS = ['disposition', 'ed_los_hours']
    
    # Re-extract features exactly how they were configured in previous cells
    exclude_cols = ['patient_id', TARGET_COL, 'chief_complaint_raw'] + HIGH_CARDINALITY_COLS
    features = [col for col in train_df.columns if col not in exclude_cols]

    logger.info("Extracting global feature gain matrices...")

    # DYNAMIC TYPECASTING: Ensure all text/object columns are clean category dtypes
    for col in features:
        if train_df[col].dtype == 'object' or train_df[col].dtype.name == 'category':
            train_df[col] = train_df[col].astype(str).astype('category')

    # Initialize the LightGBM Dataset with clean data types
    lgb_train_full = lgb.Dataset(train_df[features], label=train_df[TARGET_COL].values)

    # Train a single fast iteration tree group to score feature dependencies
    lgb_reg_params = {
        'objective': 'huber',
        'boosting_type': 'gbdt',
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': 8,
        'random_state': SEED,
        'verbose': -1,
        'n_jobs': -1
    }

    global_model = lgb.train(lgb_reg_params, lgb_train_full, num_boost_round=150)

    # Fetch importance and map to label spaces
    importances = global_model.feature_importance(importance_type='gain')
    importance_df = pd.DataFrame({
        'Feature': features,
        'Importance_Gain': importances
    }).sort_values(by='Importance_Gain', ascending=False).reset_index(drop=True)

    # Print the top 20 structural drivers
    print("\n" + "="*50)
    print("        🏆 TOP 20 CLINICAL FEATURE DRIVERS        ")
    print("="*50)
    for idx, row in importance_df.head(20).iterrows():
        print(f"{idx+1:>2}. {row['Feature']:<35} | Gain: {row['Importance_Gain']:.2f}")
    print("="*50)

    # Verify if any data leakage columns managed to pass through our extraction shields
    critical_leaks = [col for col in DATA_LEAKAGE_COLS if col in importance_df['Feature'].values]
    if len(critical_leaks) == 0:
        logger.info("Leakage Isolation Audit: PASSED. No post-triage metrics detected in training matrices.")
    else:
        logger.warning(f"Leakage Isolation Audit: FAILED. Detected active leakage tokens: {critical_leaks}")

    return importance_df

# Execute the interpretability step
importance_metrics = audit_feature_importance()

In [ ]:
"""
Advanced Emergency Triage AI Pipeline
Cell 10: NLP Vocabulary Mapping & Interpretability Translation
"""

@audit_pipeline_step("NLP Vocabulary Translation Audit")
def translate_nlp_features(importance_dataframe):
    # FORCE LOCAL IMPORT to prevent NameError
    from sklearn.feature_extraction.text import TfidfVectorizer
    
    logger.info("Re-indexing text vocabulary matrices to decode NLP tokens...")

    # Re-instantiate the vectorizer schema exactly as configured in Cell 4
    v_audit = TfidfVectorizer(
        max_features=250,
        stop_words='english',
        ngram_range=(1, 2),
        sublinear_tf=True
    )

    # Fit against our existing processed complaint array
    v_audit.fit(train_df['chief_complaint_raw'].fillna('unspecified').astype(str))
    vocab_terms = v_audit.get_feature_names_out()

    print("\n" + "="*60)
    print("      📖 DECODED MEDICAL TEXT TOKEN TRANSLATION REPORT      ")
    print("="*60)
    print(f"{'Feature Token':<15} | {'Real-World Clinical Keyword / Bigram String':<40}")
    print("-" * 60)

    # Filter out only the NLP rows from our feature importance table
    nlp_rows = importance_dataframe[importance_dataframe['Feature'].str.startswith('nlp_tfidf_')].copy()

    # Extract the original integer array index from the column string name
    nlp_rows['vocab_index'] = nlp_rows['Feature'].str.replace('nlp_tfidf_', '').astype(int)

    # Map the index to the true textual string tokens
    for _, row in nlp_rows.head(10).iterrows():
        idx = row['vocab_index']
        if idx < len(vocab_terms):
            true_word = vocab_terms[idx]
            print(f"{row['Feature']:<15} | {f'›› {true_word} ‹‹':<40}")

    print("="*60)

# Run the text decoding matrix translation
# Ensure importance_metrics is defined from your previous audit cell
translate_nlp_features(importance_metrics)

In [ ]:
"""
Advanced Emergency Triage AI Pipeline
Cell 11: Clinical Risk & Under-Triage Safety Audit
"""

@audit_pipeline_step("Clinical Risk & Under-Triage Safety Audit")
def run_safety_audit(true_labels, oof_predictions):
    total_cases = len(true_labels)

    # Calculate perfect agreements
    correct_matches = (true_labels == oof_predictions).sum()
    accuracy = correct_matches / total_cases

    # Under-triage: True ESI is lower (more urgent) than predicted ESI
    under_triage_mask = true_labels < oof_predictions
    under_triage_count = under_triage_mask.sum()
    under_triage_rate = under_triage_count / total_cases

    # Over-triage: True ESI is higher (less urgent) than predicted ESI
    over_triage_mask = true_labels > oof_predictions
    over_triage_count = over_triage_mask.sum()
    over_triage_rate = over_triage_count / total_cases

    # Isolate catastrophic misses (Errors off by more than 2 full triage categories)
    catastrophic_under = ((true_labels == 1) & (oof_predictions >= 4)).sum()
    catastrophic_over = ((true_labels >= 4) & (oof_predictions == 1)).sum()

    print("\n" + "="*60)
    print("        ⚠️ CLINICAL RISK & SAFETY DEPLOYMENT AUDIT        ")
    print("="*60)
    print(f"Total Evaluated Patient Pathways:     {total_cases:,}")
    print(f"Exact Clinical Agreement (Accuracy):  {accuracy*100:.2f}%")
    print("-" * 60)
    print(f"🟢 Conservative Over-Triage Rate:     {over_triage_rate*100:.2f}%  ({over_triage_count:,} cases)")
    print(f"    › Note: Safe error profile. Represents cautious resource over-allocation.")
    print(f"\n🔴 Dangerous Under-Triage Rate:      {under_triage_rate*100:.2f}%  ({under_triage_count:,} cases)")
    print(f"    › Note: High-risk error profile. Critical patients delayed in care.")
    print("-" * 60)
    print(f"🚨 Critical Under-Triage Level 1s:    {catastrophic_under} cases")
    print(f"    › Real ESI-1 (Resuscitation) misclassified as ESI-4/5.")
    print("="*60)

    if catastrophic_under == 0:
        logger.info("Safety Status: PASSED FOR CLINICAL DEPLOYMENT CONSIDERATION. Zero catastrophic under-triage events.")
    else:
        logger.warning(f"Safety Status: REJECTED. Fix threshold boundaries to eliminate the {catastrophic_under} severe under-triage failures.")

# Run the safety assessment
run_safety_audit(y_true, oof_cls)